# VSign: train trên Google Colab

Chuẩn bị (trên máy Windows, 1 lần mỗi khi sửa code / dữ liệu):
1. `python -m src.training.pack_colab` → tạo `vsign_colab.zip` (code + `index.csv` + `sequences.npz`, ~70 MB).
2. Upload `vsign_colab.zip` vào Google Drive, thư mục `MyDrive/VSign/` (tạo thư mục nếu chưa có).

Trên Colab: chạy lần lượt từng ô. Kết quả train ghi thẳng vào `MyDrive/VSign/runs/` → mất kết nối giữa chừng
thì chạy lại ô train, các lần đã xong sẽ được bỏ qua.

Colab chỉ cần `numpy` + `torch` (có sẵn), không cần mediapipe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/VSign'   # sửa nếu bạn để zip ở chỗ khác
WORK_DIR = '/content/vsign'

Giải nén code + dữ liệu vào `/content/vsign`, nối thư mục `runs/` sang Drive để kết quả không mất.

In [ ]:
import os, shutil, zipfile
shutil.rmtree(WORK_DIR, ignore_errors=True)
with zipfile.ZipFile(f'{DRIVE_DIR}/vsign_colab.zip') as z:
    z.extractall(WORK_DIR)
os.makedirs(f'{DRIVE_DIR}/runs', exist_ok=True)
os.symlink(f'{DRIVE_DIR}/runs', f'{WORK_DIR}/runs')
os.chdir(WORK_DIR)
!ls -R {WORK_DIR}/data
import torch, numpy, platform
print('python', platform.python_version(), '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| CPU', os.cpu_count(), '| GPU', torch.cuda.is_available())

## E0. Kiểm tra code train
Model phải học thuộc được 32 mẫu (độ chính xác → 1.000). Nếu không đạt thì có lỗi, đừng chạy tiếp.

In [ ]:
!python -m src.training.train --overfit-batch

## E0 + E1
- E0: GRU (chuẩn hoá theo vai, có z, 30 frame) và mốc MLP, seed 0.
- E1: so sánh 3 kiểu chuẩn hoá (none / shoulder / shoulder_wrist) × 3 seed.

Mặc định chạy trên CPU (`--device cpu`) để kết quả lặp lại được và so được với máy Windows.
Colab miễn phí chỉ có 2 CPU → mỗi lần train ~2-4 phút. Muốn nhanh hơn: Runtime → Change runtime type → GPU,
rồi thêm `--device cuda` (kết quả lệch nhẹ so với CPU, nên đừng trộn kết quả CPU và GPU trong cùng 1 bảng).

In [ ]:
!python -m src.training.experiments E0 E1 --device cpu --threads 2

## Bảng kết quả

In [ ]:
!python -m src.training.experiments --summary E0 E1

## Đường học (learning curve) của từng lần chạy
Train tăng mà val đứng yên/giảm = học thuộc.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
logs = sorted(glob.glob('runs/*/seed0/log.csv'))
fig, axes = plt.subplots(len(logs), 2, figsize=(11, 3 * len(logs)), squeeze=False)
for (ax_loss, ax_acc), path in zip(axes, logs):
    d = pd.read_csv(path)
    name = path.split('/')[1]
    ax_loss.plot(d.epoch, d.train_loss, label='train'); ax_loss.plot(d.epoch, d.val_loss, label='val')
    ax_loss.set_title(f'{name}: loss'); ax_loss.legend()
    ax_acc.plot(d.epoch, d.train_top1, label='train (có augmentation)'); ax_acc.plot(d.epoch, d.val_top1, label='val')
    ax_acc.set_title(f'{name}: top-1'); ax_acc.legend(); ax_acc.set_ylim(0, 1)
plt.tight_layout()

## Tiếp theo (khi đã xem xong E1)
Mỗi thí nghiệm tự lấy cấu hình thắng của thí nghiệm trước. Bỏ dấu `#` để chạy.

Test chỉ chạy 1 lần cho cấu hình CUỐI CÙNG, ví dụ:
`!python -m src.training.train --norm shoulder --seeds 0 1 2 --test`

In [ ]:
# !python -m src.training.experiments E2 E3 E4 E5 E6 --device cpu --threads 2